In [1]:
import pandas as pd

In [3]:
df_orders = pd.read_parquet("../cleaned_data/orders.parquet")
df_users = pd.read_parquet("../cleaned_data/users.parquet")

In [6]:
# Calculate first and last order date for each user
df_order_dates = (
    df_orders.groupby("user_id")
    .agg(first_order_date=("created_at", "min"), last_order_date=("created_at", "max"))
    .reset_index()
)

# Get the latest order date overall
overall_last_order = df_orders["created_at"].max()

# Calculate days since last order
df_order_dates["days_since_last_order"] = (
    overall_last_order - df_order_dates["last_order_date"]
).dt.days

# Merge with users table
df_users = df_users.merge(df_order_dates, left_on="id", right_on="user_id", how="left")

# Drop duplicate user_id column
df_users.drop(columns=["user_id"], inplace=True)


/tmp/ipykernel_53758/341455060.py:20: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_users.drop(columns=["user_id"], inplace=True)


In [7]:
df_order_dates

,user_id,first_order_date,last_order_date,days_since_last_order
0,1,2019-06-04 10:10:00+00:00,2019-06-04 10:10:00+00:00,1100
1,10,2022-04-02 01:48:00+00:00,2022-06-05 01:48:00+00:00,3
2,1000,2022-04-13 09:04:00+00:00,2022-04-13 09:04:00+00:00,56
3,10000,2021-08-03 17:13:00+00:00,2022-06-06 17:13:00+00:00,2
4,100000,2022-05-02 00:16:00+00:00,2022-05-02 00:16:00+00:00,37
...,...,...,...,...
79889,99994,2019-12-10 01:48:00+00:00,2020-05-05 01:48:00+00:00,764
79890,99995,2021-01-22 05:13:00+00:00,2021-07-11 05:13:00+00:00,332
79891,99996,2022-04-17 03:17:00+00:00,2022-04-17 03:17:00+00:00,52
79892,99997,2022-03-28 17:19:00+00:00,2022-03-28 17:19:00+00:00,72


In [8]:
df_order_dates.to_parquet("../cleaned_data/customer_activity.parquet", index=False)
